# FWIGAN: bounded package workflow

Default: tiny CPU experiment. Choose an explicit `configs/marmousi-*.yaml` path only with resources for the original expensive experiment. The package owns solver and training code; original extracted cells remain frozen in `tests/reference/`. See the standalone README for setup, units, objective and limitations.

In [ ]:
from pathlib import Path
import torch
from fwigan.cli import load_config, prepare_data
from fwigan.experiment import Experiment
from fwigan.plotting import plot_results

ROOT = Path.cwd()
if not (ROOT / "configs").is_dir():
    ROOT = ROOT.parent
CONFIG = ROOT / "configs" / "tiny-fwigan.yaml"
workflow = load_config(CONFIG)
data = prepare_data(workflow)
run = Experiment.create(workflow.experiment, data, ROOT / "runs" / "notebooks")
initial_metrics = run.evaluate()
print(initial_metrics)

## Train and inspect raw-amplitude results

This bounded default completes two tiny epochs. `max_batches` limits additional GAN shot groups or complete FWI epochs. FWIGAN noise is in normalized-data units. Evaluation uses deterministic raw pressure amplitudes and never samples learned noise.

In [ ]:
run.run(max_batches=4)
print(run.run_dir, run.position)
print(run.evaluate())
result = torch.load(run.export_final(), map_location="cpu")
figures = plot_results(data, result)

## Resume the same run

The full checkpoint restores optimizers, critic, scaler, RNG and the next position. Change settings with a new run or `Experiment.warm_start`, not by editing a saved run.

In [ ]:
continued = Experiment.resume(run.run_dir / "checkpoint.pt", device=workflow.experiment.device)
print(continued.position)
print(continued.evaluate())